# 🧠 Guri Level 1 — Train Your Own AI Brain (from scratch!)

**Welcome, builder!** In this notebook you will:

1. 🍽️ Feed free books to a brand-new brain
2. 🧱 Build the brain yourself (millions of tiny knobs, all random at first)
3. 🏋️ Train it on a **free Google GPU** (no money, no coding)
4. 💬 Chat with it!

### Before you press anything:
- **Runtime → Change runtime type → Hardware accelerator → GPU → Save**
  (this gives you a free NVIDIA GPU for a while)
- Then press **Run** on each cell **in order** (click the ▶ play button)
- A cell shows `[*]` while it works and `[1]`, `[2]`… when finished
- If a cell shows an error in red — scroll to the bottom of this guide file
  (`GUIDE.md` in the repository) and check **Troubleshooting**

> **Rule:** never close this browser tab while a cell is running!
> Google may disconnect you after ~12 hours — that's normal.
> Your brain saves checkpoints, so you can always continue later.

In [ ]:
# STEP 0 — Check that our free GPU is present
import torch

print("PyTorch:", torch.__version__)
if torch.cuda.is_available():
    print("✅ GPU:", torch.cuda.get_device_name(0))
    print("   VRAM:", round(torch.cuda.get_device_properties(0).total_mem / 1e9, 1), "GB")
else:
    print("❌ NO GPU FOUND")
    print("   Fix: Runtime → Change runtime type → Hardware accelerator → GPU → Save")
    print("   Then Runtime → Restart runtime, and start again from this cell.")

## 🍽️ Step 1 — Brain food

A brain learns by **reading**. We download free, old books (public domain —
legal for anyone to use). The more food, the smarter the brain.

In [ ]:
# STEP 1 — Get the brain's food (free public-domain books from Project Gutenberg)
import urllib.request
from pathlib import Path

DATA_DIR = Path("/content/guri_data")
DATA_DIR.mkdir(parents=True, exist_ok=True)

# How much food: "small" = 1 book (~15 min of training data)
#                 "normal" = 5 books (recommended)
book_amount = "normal"  # @param ["small", "normal"]

BOOKS = {
    "small": [11],                       # Alice in Wonderland
    "normal": [11, 74, 1342, 84, 2701],  # Alice, Sherlock Holmes, Pride and Prejudice, Frankenstein, Moby Dick
}[book_amount]

def fetch(book_id: int) -> str | None:
    urls = [
        f"https://www.gutenberg.org/cache/epub/{book_id}/pg{book_id}.txt",
        f"https://www.gutenberg.org/files/{book_id}/{book_id}-0.txt",
        f"https://www.gutenberg.org/files/{book_id}/{book_id}.txt",
    ]
    for url in urls:
        try:
            req = urllib.request.Request(url, headers={"User-Agent": "GuriBrain/0.1 (training)"})
            with urllib.request.urlopen(req, timeout=30) as r:
                text = r.read().decode("utf-8", errors="ignore")
            if len(text) > 10_000 and not text.lstrip().startswith("<!"):
                return text
        except Exception as e:  # noqa: BLE001 — try the next mirror
            print(f"   (mirror failed for {book_id}: {type(e).__name__})")
    return None

paths = []
for bid in BOOKS:
    dest = DATA_DIR / f"book_{bid}.txt"
    if not dest.exists() or dest.stat().st_size < 10_000:
        print(f"Downloading book {bid} ...")
        text = fetch(bid)
        if text is None:
            print(f"   ❌ Could not download book {bid} — trying the next one.")
            continue
        dest.write_text(text, encoding="utf-8")
    print(f"   ✅ {dest.name}: {dest.stat().st_size:,} characters")
    paths.append(str(dest))

if not paths:
    print("⚠️  No internet books available — using the bundled mini story pack.")
    print("   (Training will work, but a bigger dinner makes a smarter brain.)")
    fallback = DATA_DIR / "sample.txt"
    fallback.write_text("The Tiny Village of Lanterns\n\nIn a tiny village between two green hills, there lived a girl named Mira.\nMira had a red scarf, a curious smile, and a little lamp that never went\nout. Every evening she walked the muddy path and lit the lanterns of her\nneighbors. \"Good evening,\" she would say. \"Your light is waiting.\"\n\nOne night the wind blew hard and the river rose. The bridge shook, and the\nold baker cried out, \"My bread! The water will take my bread!\" Mira held\nher lamp high and waded into the cold water. She pulled the bread baskets\nto the hill, one by one, until the last loaf was safe.\n\nIn the morning the villagers clapped and cheered. The baker gave Mira a\nwarm loaf with honey on top. \"You are small,\" he said, \"but your light is\nbig.\" Mira laughed and wrapped her red scarf tight.\n\nFrom that day on, when the wind came, the villagers were not afraid. They\nwould light every lantern and wait for Mira's smile on the hill.\n\n\nThe Robot Who Wanted a Name\n\nIn a workshop full of gears and wires, a small robot opened his eyes for\nthe first time. \"I am ready,\" he said. \"But I have no name.\"\n\nHe asked the clock on the wall. The clock said, \"Tick tock, I only know\ntime.\" He asked the paint bucket. The paint bucket said, \"Splash, I only\nknow colors.\" He asked the cat sleeping by the door. The cat opened one\neye and said, \"Purr. You look like a Bolt to me.\"\n\nThe robot smiled. \"Bolt,\" he said. \"I like that.\"\n\nEvery day Bolt helped the workers carry heavy boxes. He never complained\nwhen his arms grew tired. When the lights went out, he glowed softly so\nthe workers could see. \"Thank you, Bolt,\" they said.\n\nOne day a little girl visited the workshop and dropped her teddy bear.\nBolt found it under the big machine and carried it back with both hands.\nThe girl hugged the bear and then hugged Bolt. \"You are kind,\" she said.\n\nBolt did not need a crown or a medal. He had a name, a job, and a friend.\nThat was enough for any robot in the whole wide world.\n\n\nThe River and the Stone\n\nA smooth gray stone sat in the middle of a bright river. Years passed.\nThe water pushed and pushed, round and soft, without any anger.\n\n\"Stop moving me!\" cried the stone one day. \"I am fine where I am.\"\n\nThe river answered in a quiet voice, \"I am not your enemy. I am teaching\nyou.\" The stone did not understand, so it waited.\n\nWinter came, then spring, then summer again. Slowly the sharp corners of\nthe stone grew round. Boats passed over it without scratching. Children\nsat on it to fish. Birds landed on it to drink.\n\nMany years later the stone was smooth as an egg. \"Now I see,\" it said.\n\"You were not moving me to be mean. You were making me ready for the\nworld.\" The river just kept flowing, as rivers do.\n\nMoral of the story: patience and kindness can shape us into something\nbeautiful, even when the work feels hard.\n\n\nA Letter to the Moon\n\nDear Moon,\n\nTonight you are full and bright, like a lamp in the sky. I am writing to\nyou from my window. My homework is done, my dog is asleep, and the house\nis very quiet.\n\nI want to ask you a question. Do you ever get lonely up there, all by\nyourself? When I feel lonely, I talk to my sister or I draw a picture.\nWhat do you do?\n\nMy teacher says you do not make your own light. You only borrow the sun's\nlight and share it with us. I think that is a very nice thing to do. If I\nborrowed something, I would share it too.\n\nPlease keep shining. I will keep watching from my window. Maybe one day I\nwill visit you and we can look down at the Earth together. It must look\nsmall from up there, like a blue marble in a jar.\n\nGood night, Moon. Thank you for the light.\n\nYour friend,\nAsha\n\n\nThe Baker's First Loaf\n\nThe baker had made ten thousand loaves of bread, but he had never once\nbaked a loaf for himself. Every morning he rose before the sun. He\nkneaded the dough, watched the oven, and gave the first warm loaf to\nwhoever stood at his door.\n\nOne winter morning an old woman came slowly down the street. She had no\nmoney in her hand. The baker wrapped a loaf in a clean cloth and pressed\nit into her fingers. \"It is free today,\" he said softly.\n\nThe woman smiled. \"Why are you so kind?\" she asked.\n\nThe baker thought for a moment. \"Because bread tastes better when it is\nshared,\" he said.\n\nThat night the baker mixed one small batch of dough, just for himself.\nHe waited. He watched. When the loaf came out golden, he cut it in half\nand gave one piece to the cat by the oven.\n\n\"This,\" he said, \"is the best loaf of all.\"\n\n\nTwo Friends and the Kite\n\nSam and Leena were best friends. They built a kite together out of old\nnewspaper, two sticks, and a long red tail. \"It will fly,\" Sam said. \"It\nmust fly,\" Leena said.\n\nThey ran to the hill. The wind was shy that day. The kite bumped the\ngrass and fell. They tried again. It rose a little, then tumbled down.\n\nLeena wanted to cry, but she did not. \"Once more,\" she said. They ran\nfaster this time, and Leena let go at exactly the right moment. The kite\nsoared up, up, up into the blue.\n\n\"Look!\" shouted Sam. \"It is higher than the birds!\"\n\nThey took turns holding the string until the sun went pink. Walking home,\nLeena said, \"We did it together.\" Sam nodded. \"Together,\" he said, \"is\nthe best way to do anything.\"\n\n\nThe Sleepy Train\n\nAt midnight the little train woke up on the tracks. \"Where is everybody?\"\nhe asked. The station was empty. The lamps were dim. Even the ticket\nman was snoring in his chair.\n\n\"I will make one more run,\" said the train. \"Just in case someone needs\nme.\" He puffed past the dark farms and the quiet river. He passed the\nhill where the sheep slept like white clouds.\n\nHalfway up the mountain he found a boy sitting on a suitcase. The boy\nwas cold and tears were on his cheeks. \"I missed my train,\" he sniffled.\n\n\"Hop on,\" said the train kindly. \"No one is left behind on my watch.\"\n\nHe carried the boy all the way home to the bright city station. The boy's\nmother was waiting with open arms. \"Thank you,\" she whispered.\n\nThe train clicked back to his shed before sunrise, tired but happy. A\ntrain with a good heart never sleeps too long.\n\n\nThe Garden on the Roof\n\nGrandfather had a garden on the roof: tomatoes, mint, and yellow flowers\nthat turned to face the sun. Every morning he climbed the stairs with a\nsteel watering can, and every morning his granddaughter Nima followed\nbehind.\n\n\"Water only the roots,\" Grandfather said. \"Leaves can drink the rain.\"\n\nNima asked, \"Why do we grow food on a roof? There is plenty of ground\nbelow.\"\n\nGrandfather smiled. \"The sky is hungry too,\" he said. \"And a garden\nteaches patience. You plant today, you eat next month.\"\n\nIn summer the tomatoes came in red and round. Nima bit into one and\njuice ran down her chin. \"Sweet!\" she cried.\n\nGrandfather laughed. \"That is the taste of waiting well.\"\n\nOn cold nights they sat by the window and planned next year's garden.\nThe roof was small, but their dreams were not.\n\n\nThe Boy Who Drew Maps\n\nIn a classroom by the sea, a boy named Tenzin drew maps all day. He drew\nthe way to the bakery, the way to the river, and the way to his\ngrandmother's house with the blue door.\n\nHis teacher said, \"Maps are for big roads and far countries.\" Tenzin\nnodded but kept drawing. He drew a map of the school hallway so the new\nstudent would not get lost. He drew a map of the garden so the bees could\nfind the flowers, though he knew bees had their own ideas.\n\nOne day the town held a festival. Someone misplaced the tent stakes, and\nthe flags drooped in the mud. Tenzin ran home and returned with a careful\nmap of the field: where the stage should stand, where the lights should\nhang, where the children should sit.\n\nBy sunset the festival shone like a star. The mayor shook Tenzin's hand.\n\"You see the world in paths,\" she said.\n\nTenzin smiled and turned to a fresh page. There was always another road\nto draw, another friend to help along the way.\n\n\nHow the Stars Learned to Shine\n\nLong ago the sky was dark and the stars were only stones floating in the\nblack. They grumbled and bumped into one another. \"It is too dark to\nsee,\" they said. \"Nobody knows we are here.\"\n\nAn old cloud drifted past. \"Then make some light,\" she said softly.\n\nThe smallest stone tried first. She held one tiny spark deep inside\nherself and would not let it go. Slowly the spark grew warm, then bright,\nthen golden. The other stones watched in wonder.\n\n\"One by one,\" said the cloud, \"or not at all.\"\n\nSo each stone lit its own small fire. From far below, people looked up\nand smiled. Children named them. Lovers wished on them. Sailors used\nthem to find their way home.\n\nThe sky is never truly dark now. It only waits for each of us to find\nthe light we carry, and to keep it burning for someone else.\n", encoding="utf-8")
    paths = [str(fallback)]

print(f"\n🍽️  {len(paths)} food file(s) ready.")
data_paths_cell = paths  # used by the training step later

## 🧱 Step 2 — Settings + the mini-dictionary

The brain doesn't read letters — it reads **numbers**. So we build a
mini-dictionary (called a *tokenizer*) that converts words into numbers.

Then you choose **how long** to train. Leave it on `normal` the first time.

In [ ]:
# STEP 2 — SETTINGS (this is the only cell you should ever edit!)
#
# The brain's report card number is called "loss":
#   loss starts HIGH (confused) and we want it to go DOWN (smarter).

SAVE_TO_GOOGLE_DRIVE = True  # @param {type:"boolean"}
HOW_LONG = "normal"  # @param ["quick", "normal", "long"]

# Training length presets (a "step" = one tiny lesson for the brain)
PRESETS = {
    "quick":  1500,   # ~10-15 minutes on the free GPU  (silliest brain)
    "normal": 5000,   # ~30-50 minutes                  (recommended!)
    "long":   15000,  # many hours, resumes across sessions (smartest)
}
MAX_STEPS = PRESETS[HOW_LONG]

if SAVE_TO_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    OUT_DIR = "/content/drive/MyDrive/GuriLevel1"
else:
    OUT_DIR = "/content/guri_out"

import os
os.makedirs(OUT_DIR, exist_ok=True)

CFG = {
    "data_paths": data_paths_cell,
    "out_dir": OUT_DIR,
    "vocab_path": os.path.join(OUT_DIR, "vocab.json"),
    "max_steps": MAX_STEPS,
}
print(f"✅ Settings ready: {MAX_STEPS:,} steps -> {OUT_DIR}")

In [ ]:
"""
Guri Level 1 data tools — the brain's food pipeline.

1. Read plain text files.
2. Build a mini-dictionary (tokenizer) FROM our own text — not downloaded.
3. Turn text into number lists the brain can eat.
"""

import json
import re
from collections import Counter
from pathlib import Path

import numpy as np
import torch

# Special "meta" tokens every brain needs.
PAD, UNK, BOS, EOS = "<pad>", "<unk>", "<bos>", "<eos>"
SPECIAL_TOKENS = [PAD, UNK, BOS, EOS]

# Split text into words and punctuation marks: "Hello, world!" -> [hello, ,, world, !]
_TOKEN_RE = re.compile(r"\w+|[^\w\s]")


def split_tokens(text: str) -> list[str]:
    """Lowercase and chop text into little pieces (words + punctuation)."""
    return _TOKEN_RE.findall(text.lower())


class SimpleTokenizer:
    """A from-scratch word-level tokenizer (our own mini-dictionary)."""

    def __init__(self, token_to_id: dict[str, int]):
        self.token_to_id = token_to_id
        self.id_to_token = {i: t for t, i in token_to_id.items()}

    @classmethod
    def build(cls, text: str, max_vocab: int = 16000, min_freq: int = 2) -> "SimpleTokenizer":
        """Grow the dictionary from the words that actually appear in our food."""
        counts = Counter(split_tokens(text))
        token_to_id = {t: i for i, t in enumerate(SPECIAL_TOKENS)}
        for token, freq in counts.most_common():
            if freq < min_freq or len(token_to_id) >= max_vocab:
                break
            token_to_id[token] = len(token_to_id)
        return cls(token_to_id)

    @property
    def vocab_size(self) -> int:
        return len(self.token_to_id)

    def encode(self, text: str, add_eos: bool = False) -> list[int]:
        ids = [self.token_to_id.get(t, self.token_to_id[UNK]) for t in split_tokens(text)]
        if add_eos:
            ids.append(self.token_to_id[EOS])
        return ids

    def decode(self, ids) -> str:
        words = []
        for i in ids:
            tok = self.id_to_token.get(int(i), UNK)
            if tok in (PAD, BOS, EOS):
                if tok == EOS:
                    words.append("\n")
                continue
            words.append(tok)
        # Glue words back together with spaces; keep punctuation snug.
        out = ""
        for w in words:
            if not out:
                out = w
            elif re.fullmatch(r"[^\w]", w):
                out += w
            else:
                out += " " + w
        return out

    def save(self, path: str | Path) -> None:
        Path(path).write_text(json.dumps(self.token_to_id, ensure_ascii=False, indent=0))

    @classmethod
    def load(cls, path: str | Path) -> "SimpleTokenizer":
        return cls(json.loads(Path(path).read_text()))


def build_corpus(paths: list[str | Path], max_files: int | None = None) -> str:
    """Read and stitch text files into one big scroll of brain food."""
    chunks = []
    for p in (paths[:max_files] if max_files else paths):
        text = Path(p).read_text(encoding="utf-8", errors="ignore")
        chunks.append(text.strip())
    return "\n\n".join(chunks)


def encode_files(paths: list[str | Path], tokenizer: SimpleTokenizer) -> np.ndarray:
    """Turn every file into token numbers, one after another on a long tape.

    Each file ends with a real EOS token so the brain learns where stories end.
    """
    ids: list[int] = []
    for p in paths:
        text = Path(p).read_text(encoding="utf-8", errors="ignore")
        ids.extend(tokenizer.encode(text, add_eos=True))
    return np.asarray(ids, dtype=np.int32)


def train_val_split(tokens: np.ndarray, val_fraction: float = 0.03):
    """Keep a small slice of tokens aside as a 'practice test' set."""
    n_val = max(1, int(len(tokens) * val_fraction))
    if len(tokens) - n_val <= 1:
        raise ValueError("Not enough data to train — add more text files.")
    return tokens[:-n_val], tokens[-n_val:]


def get_batch(
    data: np.ndarray, block_size: int, batch_size: int, device: torch.device
) -> tuple[torch.Tensor, torch.Tensor]:
    """Grab random windows of text — the brain's snack for one step."""
    starts = np.random.randint(0, len(data) - block_size - 1, size=batch_size)
    x = np.stack([data[s : s + block_size] for s in starts])
    y = np.stack([data[s + 1 : s + 1 + block_size] for s in starts])
    return (
        torch.from_numpy(x.astype(np.int64)).to(device),
        torch.from_numpy(y.astype(np.int64)).to(device),
    )

## 🔢 Step 3 — Turn the books into a number tape

In [ ]:
# Build the mini-dictionary from OUR books (not downloaded from anyone!)
tokenizer = SimpleTokenizer.build(
    build_corpus(data_paths_cell),
    max_vocab=CFG.get("max_vocab", 16000),
    min_freq=2,
)
tokenizer.save(CFG["vocab_path"])
print(f"📖 Mini-dictionary: {tokenizer.vocab_size:,} words/pieces")

tokens = encode_files(data_paths_cell, tokenizer)
train_data, val_data = train_val_split(tokens)
print(f"🎞️  Number tape: {len(tokens):,} tokens")
print(f"   train: {len(train_data):,}   practice-test: {len(val_data):,}")

# Show a tiny example so it feels real
example = "The robot smiled and waved."
ids = tokenizer.encode(example)
print(f'\nExample: "{example}" -> {ids[:12]}...')
print(f"Back to words: {tokenizer.decode(ids)}")

## 🧱 Step 4 — Build the brain

Same idea as GPT, but mini-sized: about **8–9 million knobs** (a real 3-billion
brain would need a supercomputer — we level up to that later!).

In [ ]:
"""
Guri Level 1 Brain — a tiny GPT-style language model built FROM SCRATCH.

No downloaded brains. No pre-trained weights. Every number in here starts
random and gets smarter only by training on text we feed it.

The math is the same family as GPT, just much smaller (about 8-9 million
"knobs" instead of billions).
"""

from dataclasses import dataclass
import math

import torch
import torch.nn as nn
from torch.nn import functional as F


@dataclass
class GPTConfig:
    """Settings for the brain. Bigger numbers = bigger brain = slower training."""

    vocab_size: int = 16000   # size of the mini-dictionary (set from data.py)
    block_size: int = 512     # how many tokens the brain looks at once (memory span)
    n_layer: int = 6          # how many transformer layers (depth)
    n_head: int = 4           # how many attention "spotlights" per layer
    n_embd: int = 256         # size of each number-vector (width)
    dropout: float = 0.1      # forgetting noise that prevents memorizing


class CausalSelfAttention(nn.Module):
    """Each token looks at earlier tokens and decides what matters.

    'Causal' means it can only look backwards (into the past), never at
    future words — that's how it learns to predict what comes next.
    """

    def __init__(self, cfg: GPTConfig):
        super().__init__()
        assert cfg.n_embd % cfg.n_head == 0
        self.n_head = cfg.n_head
        self.n_embd = cfg.n_embd
        self.c_attn = nn.Linear(cfg.n_embd, 3 * cfg.n_embd, bias=False)  # query, key, value
        self.c_proj = nn.Linear(cfg.n_embd, cfg.n_embd, bias=False)
        self.attn_dropout = nn.Dropout(cfg.dropout)
        self.resid_dropout = nn.Dropout(cfg.dropout)
        # The "wall" that hides the future: a lower-triangular mask.
        self.register_buffer(
            "mask",
            torch.tril(torch.ones(cfg.block_size, cfg.block_size))
            .view(1, 1, cfg.block_size, cfg.block_size),
            persistent=False,
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        B, T, C = x.size()  # batch, sequence length, embedding size
        q, k, v = self.c_attn(x).split(self.n_embd, dim=2)
        # Split heads: (B, T, C) -> (B, n_head, T, head_dim)
        q = q.view(B, T, self.n_head, C // self.n_head).transpose(1, 2)
        k = k.view(B, T, self.n_head, C // self.n_head).transpose(1, 2)
        v = v.view(B, T, self.n_head, C // self.n_head).transpose(1, 2)

        att = (q @ k.transpose(-2, -1)) / math.sqrt(k.size(-1))
        att = att.masked_fill(self.mask[:, :, :T, :T] == 0, float("-inf"))
        att = F.softmax(att, dim=-1)
        att = self.attn_dropout(att)
        y = att @ v
        y = y.transpose(1, 2).contiguous().view(B, T, C)
        return self.resid_dropout(self.c_proj(y))


class MLP(nn.Module):
    """The 'thinking' layer: expand each token's info, mash it, squeeze back."""

    def __init__(self, cfg: GPTConfig):
        super().__init__()
        self.c_fc = nn.Linear(cfg.n_embd, 4 * cfg.n_embd, bias=False)
        self.c_proj = nn.Linear(4 * cfg.n_embd, cfg.n_embd, bias=False)
        self.dropout = nn.Dropout(cfg.dropout)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.dropout(self.c_proj(F.gelu(self.c_fc(x))))


class Block(nn.Module):
    """One layer of the brain: look around (attention), then think (MLP)."""

    def __init__(self, cfg: GPTConfig):
        super().__init__()
        self.ln_1 = nn.LayerNorm(cfg.n_embd)
        self.attn = CausalSelfAttention(cfg)
        self.ln_2 = nn.LayerNorm(cfg.n_embd)
        self.mlp = MLP(cfg)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = x + self.attn(self.ln_1(x))  # residual: keep the old info too
        x = x + self.mlp(self.ln_2(x))
        return x


class GPT(nn.Module):
    """The whole brain: turns a list of token numbers into next-token guesses."""

    def __init__(self, cfg: GPTConfig):
        super().__init__()
        self.cfg = cfg
        self.tok_emb = nn.Embedding(cfg.vocab_size, cfg.n_embd)
        self.pos_emb = nn.Embedding(cfg.block_size, cfg.n_embd)
        self.drop = nn.Dropout(cfg.dropout)
        self.blocks = nn.ModuleList([Block(cfg) for _ in range(cfg.n_layer)])
        self.ln_f = nn.LayerNorm(cfg.n_embd)
        self.lm_head = nn.Linear(cfg.n_embd, cfg.vocab_size, bias=False)

        # Weight tying: the brain uses the same numbers to read and to write.
        self.lm_head.weight = self.tok_emb.weight

        self.apply(self._init_weights)
        # Special starting scale for the residual projections (like GPT-2).
        for name, p in self.named_parameters():
            if name.endswith("c_proj.weight"):
                nn.init.normal_(p, mean=0.0, std=0.02 / math.sqrt(2 * cfg.n_layer))

    @staticmethod
    def _init_weights(module: nn.Module) -> None:
        if isinstance(module, nn.Linear):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def num_parameters(self) -> int:
        """How many 'knobs' does this brain have? (embedding counted once)"""
        n = sum(p.numel() for p in self.parameters())
        n -= self.lm_head.weight.numel()  # tied with tok_emb, don't double count
        return n

    def forward(self, idx: torch.Tensor, targets: torch.Tensor | None = None):
        B, T = idx.size()
        assert T <= self.cfg.block_size, f"sequence too long: {T}"
        pos = torch.arange(0, T, dtype=torch.long, device=idx.device)

        x = self.drop(self.tok_emb(idx) + self.pos_emb(pos))
        for block in self.blocks:
            x = block(x)
        x = self.ln_f(x)
        logits = self.lm_head(x)

        loss = None
        if targets is not None:
            loss = F.cross_entropy(
                logits.view(-1, logits.size(-1)), targets.view(-1), ignore_index=-1
            )
        return logits, loss

    @torch.no_grad()
    def generate(
        self,
        idx: torch.Tensor,
        max_new_tokens: int = 200,
        temperature: float = 0.8,
        top_k: int | None = 50,
    ) -> torch.Tensor:
        """Dream up new tokens one at a time (sampling with temperature/top-k)."""
        self.eval()
        for _ in range(max_new_tokens):
            idx_cond = idx[:, -self.cfg.block_size :]
            logits, _ = self(idx_cond)
            logits = logits[:, -1, :] / max(temperature, 1e-6)
            if top_k is not None:
                v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
                logits[logits < v[:, [-1]]] = float("-inf")
            probs = F.softmax(logits, dim=-1)
            next_id = torch.multinomial(probs, num_samples=1)
            idx = torch.cat((idx, next_id), dim=1)
        return idx

In [ ]:
cfg_model = GPTConfig(
    vocab_size=tokenizer.vocab_size,
    block_size=512,
    n_layer=6,
    n_head=4,
    n_embd=256,
    dropout=0.1,
)
model = GPT(cfg_model)
n = model.num_parameters()
print(f"🧠 Your brain has {n:,} knobs ({n/1e6:.2f} million)")
print("   All of them are RANDOM right now. Training will teach them.")
print("   It knows nothing yet — like a baby who never heard a word.")

## 🏋️ Step 5 — Train! (watch the loss go down)

- **loss = the brain's report card.** High = confused, low = smart.
- The first numbers will be around **10** (pure confusion). We want **under 5**.
- Checkpoints save automatically. If Colab disconnects: **Runtime → Reconnect**,
  re-run the cells above (they'll reload from Drive), and training **resumes**.

In [ ]:
"""
Guri Level 1 trainer — teaches the brain by showing it millions of examples.

Design rules:
  * Works on free Google Colab (T4 GPU) and on plain CPU (slow, for testing).
  * Saves checkpoints often, so a kicked session never loses progress.
  * Resumes automatically: re-run and it picks up where it left off.
"""

import math
import time
from pathlib import Path

import numpy as np
import torch


# ---------------------------------------------------------------------------
# EDIT ME: the knobs at the top of the recipe. Kids: this is the only
# section you ever need to touch.
# ---------------------------------------------------------------------------
DEFAULT_CONFIG = {
    "data_paths": ["data/sample.txt"],   # where the brain food lives
    "out_dir": "out",                    # where checkpoints get saved
    "vocab_path": "out/vocab.json",      # where the mini-dictionary is saved
    "max_vocab": 16000,                  # mini-dictionary size
    "min_freq": 2,                       # ignore ultra-rare words
    "block_size": 512,                   # memory span (tokens)
    "batch_size": 32,                    # snacks per step
    "n_layer": 6,                        # depth of the brain
    "n_head": 4,                         # attention spotlights
    "n_embd": 256,                       # width of the brain
    "dropout": 0.1,
    "learning_rate": 3e-4,               # how big each correction is
    "min_lr_ratio": 0.1,                 # LR decays to 10% of starting value
    "warmup_steps": 200,                 # gentle start
    "max_steps": 5000,                   # total training steps
    "eval_interval": 250,                # how often to measure progress
    "eval_batches": 40,                  # practice-test batches per eval
    "save_interval": 500,                # checkpoint every N steps
    "grad_clip": 1.0,                    # stop crazy big corrections
    "seed": 42,
    "device": "auto",                    # "auto" picks GPU if available
}


def pick_device(name: str = "auto") -> torch.device:
    if name != "auto":
        return torch.device(name)
    if torch.cuda.is_available():
        return torch.device("cuda")
    print("⚠️  No GPU found — training on CPU. It works, but much slower.")
    print("   In Colab: Runtime -> Change runtime type -> GPU, then re-run.")
    return torch.device("cpu")


def lr_at(step: int, cfg: dict) -> float:
    """Warm up gently, then decay — like easing into a cold pool."""
    if step < cfg["warmup_steps"]:
        return cfg["learning_rate"] * (step + 1) / max(1, cfg["warmup_steps"])
    progress = (step - cfg["warmup_steps"]) / max(1, cfg["max_steps"] - cfg["warmup_steps"])
    progress = min(max(progress, 0.0), 1.0)
    min_lr = cfg["learning_rate"] * cfg["min_lr_ratio"]
    return min_lr + 0.5 * (cfg["learning_rate"] - min_lr) * (1 + math.cos(math.pi * progress))


@torch.no_grad()
def estimate_loss(model, data, cfg, device) -> float:
    """Score the brain on held-out text it never trained on (practice test)."""
    model.eval()
    losses = []
    for _ in range(cfg["eval_batches"]):
        x, y = get_batch(data, cfg["block_size"], cfg["batch_size"], device)
        with torch.autocast(device_type=device.type, enabled=device.type == "cuda"):
            _, loss = model(x, y)
        losses.append(loss.item())
    model.train()
    return float(np.mean(losses))


def save_checkpoint(path: Path, model, optimizer, scaler, step: int, cfg: dict) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    tmp = path.with_suffix(".tmp")
    torch.save(
        {
            "model": model.state_dict(),
            "optimizer": optimizer.state_dict(),
            "scaler": scaler.state_dict() if scaler is not None else None,
            "step": step,
            "model_config": {
                "vocab_size": model.cfg.vocab_size,
                "block_size": model.cfg.block_size,
                "n_layer": model.cfg.n_layer,
                "n_head": model.cfg.n_head,
                "n_embd": model.cfg.n_embd,
                "dropout": model.cfg.dropout,
            },
            "train_config": cfg,
        },
        tmp,
    )
    tmp.replace(path)  # atomic-ish: never leave a half-written checkpoint


def train(cfg: dict | None = None) -> dict:
    """Main training loop. Returns a small report dict."""
    cfg = {**DEFAULT_CONFIG, **(cfg or {})}
    torch.manual_seed(cfg["seed"])
    np.random.seed(cfg["seed"])
    device = pick_device(cfg["device"])
    out_dir = Path(cfg["out_dir"])
    out_dir.mkdir(parents=True, exist_ok=True)

    # ---- Brain food -------------------------------------------------------
    print("🍽️  Reading brain food...")
    corpus = build_corpus(cfg["data_paths"])
    print(f"   {len(corpus):,} characters from {len(cfg['data_paths'])} file(s)")

    Path(cfg["vocab_path"]).parent.mkdir(parents=True, exist_ok=True)
    tokenizer = SimpleTokenizer.build(
        corpus, max_vocab=cfg["max_vocab"], min_freq=cfg["min_freq"]
    )
    tokenizer.save(cfg["vocab_path"])
    print(f"   Mini-dictionary: {tokenizer.vocab_size:,} tokens -> {cfg['vocab_path']}")

    tokens = encode_files(cfg["data_paths"], tokenizer)
    train_data, val_data = train_val_split(tokens)
    print(f"   Token tape: {len(tokens):,} tokens "
          f"({len(train_data):,} train / {len(val_data):,} val)")

    # Shrink the memory span if the data is tiny (keeps small demos working).
    smallest = min(len(train_data), len(val_data))
    if smallest < cfg["block_size"] + 2:
        cfg["block_size"] = max(32, smallest - 2)
        print(f"   ⚠️  Small dataset: memory span reduced to {cfg['block_size']} tokens. "
              f"Add more text for a smarter brain!")

    # ---- Build the brain --------------------------------------------------
    model_cfg = GPTConfig(
        vocab_size=tokenizer.vocab_size,
        block_size=cfg["block_size"],
        n_layer=cfg["n_layer"],
        n_head=cfg["n_head"],
        n_embd=cfg["n_embd"],
        dropout=cfg["dropout"],
    )
    model = GPT(model_cfg).to(device)
    n_params = model.num_parameters()
    print(f"\n🧠 Brain built: {n_params / 1e6:.2f} million parameters on {device}")

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=cfg["learning_rate"],
        betas=(0.9, 0.95),
        weight_decay=0.1,
    )
    use_amp = device.type == "cuda"
    scaler = torch.amp.GradScaler("cuda", enabled=use_amp)

    # ---- Resume if a checkpoint exists ------------------------------------
    latest = out_dir / "ckpt_latest.pt"
    start_step = 0
    if latest.exists():
        blob = torch.load(latest, map_location=device, weights_only=False)
        model.load_state_dict(blob["model"])
        optimizer.load_state_dict(blob["optimizer"])
        if blob.get("scaler") and use_amp:
            scaler.load_state_dict(blob["scaler"])
        start_step = int(blob["step"])
        print(f"🔁 Resumed from step {start_step:,} (training continues!)")

    # ---- The learning loop ------------------------------------------------
    print(f"\n🚀 Training for {cfg['max_steps']:,} steps "
          f"(already done: {start_step:,}). Keep this tab open!\n")
    print(f"{'step':>7} {'train':>8} {'val':>8} {'lr':>10} {'tok/s':>9}  note")
    print("-" * 62)

    model.train()
    t0 = time.time()
    tokens_since_log = 0
    window_start = t0
    report = {"best_val": float("inf"), "final_step": start_step}

    for step in range(start_step + 1, cfg["max_steps"] + 1):
        lr = lr_at(step - 1, cfg)
        for group in optimizer.param_groups:
            group["lr"] = lr

        x, y = get_batch(train_data, cfg["block_size"], cfg["batch_size"], device)
        with torch.autocast(device_type=device.type, enabled=use_amp):
            _, loss = model(x, y)
        optimizer.zero_grad(set_to_none=True)
        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), cfg["grad_clip"])
        scaler.step(optimizer)
        scaler.update()
        tokens_since_log += x.numel()

        if step % cfg["eval_interval"] == 0 or step == cfg["max_steps"]:
            train_loss = estimate_loss(model, train_data, cfg, device)
            val_loss = estimate_loss(model, val_data, cfg, device)
            now = time.time()
            tok_s = tokens_since_log / max(now - window_start, 1e-6)
            tokens_since_log = 0
            window_start = now

            note = ""
            if val_loss < report["best_val"]:
                report["best_val"] = val_loss
                note = "⭐ new best!"
                save_checkpoint(
                    out_dir / "ckpt_best.pt", model, optimizer, scaler, step, cfg
                )
            if step % cfg["save_interval"] == 0 or step == cfg["max_steps"]:
                save_checkpoint(latest, model, optimizer, scaler, step, cfg)
                if not note:
                    note = "💾 saved"
            print(f"{step:>7,} {train_loss:>8.3f} {val_loss:>8.3f} "
                  f"{lr:>10.2e} {tok_s:>9,.0f}  {note}")

    elapsed = time.time() - t0
    save_checkpoint(latest, model, optimizer, scaler, cfg["max_steps"], cfg)
    print("\n🎓 Training finished!")
    print(f"   Time: {elapsed / 60:.1f} minutes")
    print(f"   Best practice-test score (val loss): {report['best_val']:.3f}")
    print(f"   Brain saved to: {latest}")
    print(f"   Dictionary saved to: {cfg['vocab_path']}")
    print("\n👉 Next: run the chat section and TALK TO YOUR BRAIN!")
    report["final_step"] = cfg["max_steps"]
    report["checkpoint"] = str(latest)
    return report

In [ ]:
# 🚀 START TRAINING — keep this tab open and watch the numbers!
report = train(CFG)

## 💬 Step 6 — Talk to your brain!

Type a message and press Enter. The brain continues your text —
it's like a clever parrot that read a whole library. 🦜

Commands inside chat: `/temp 0.8` (creativity), `/tokens 200` (length), `/quit`

In [ ]:
"""
Guri Level 1 chat — talk to the brain you trained.

Loads the checkpoint + dictionary, then lets you type messages and read
what the brain dreams up next.
"""

import argparse
from pathlib import Path

import torch



def load_brain(ckpt_path: str | Path, vocab_path: str | Path | None = None, device=None):
    """Load a trained brain and its dictionary from disk."""
    device = device or torch.device("cuda" if torch.cuda.is_available() else "cpu")
    blob = torch.load(ckpt_path, map_location=device, weights_only=False)
    model_cfg = GPTConfig(**blob["model_config"])
    model = GPT(model_cfg)
    model.load_state_dict(blob["model"])
    model.to(device).eval()

    if vocab_path is None:
        # Default: vocab.json sitting next to the checkpoint.
        vocab_path = Path(ckpt_path).parent / "vocab.json"
    tokenizer = SimpleTokenizer.load(vocab_path)
    print(f"🧠 Loaded brain from step {blob.get('step', '?')} "
          f"({model.num_parameters() / 1e6:.2f}M params) on {device}")
    return model, tokenizer, device


@torch.no_grad()
def reply(
    model,
    tokenizer,
    prompt: str,
    device,
    max_new_tokens: int = 150,
    temperature: float = 0.8,
    top_k: int = 50,
) -> str:
    """Feed a prompt to the brain and read what it writes back."""
    ids = tokenizer.encode(prompt)
    if not ids:
        ids = [tokenizer.token_to_id["<bos>"]]
    x = torch.tensor([ids], dtype=torch.long, device=device)
    out = model.generate(x, max_new_tokens=max_new_tokens,
                         temperature=temperature, top_k=top_k)
    new_ids = out[0, len(ids):].tolist()
    return tokenizer.decode(new_ids)


def chat_loop(model, tokenizer, device) -> None:
    """A simple command-line chat window (works in Colab too)."""
    print("\n💬 Chat with your brain! (type /quit to leave)")
    print("   Commands: /temp 0.8   /tokens 200   /prompt some text\n")
    temperature, max_new = 0.8, 150
    while True:
        try:
            you = input("You  > ").strip()
        except (EOFError, KeyboardInterrupt):
            print("\n👋 Bye!")
            break
        if not you:
            continue
        if you == "/quit":
            print("👋 Bye!")
            break
        if you.startswith("/temp"):
            try:
                temperature = float(you.split()[1])
                print(f"   (temperature = {temperature})")
            except (IndexError, ValueError):
                print("   usage: /temp 0.8")
            continue
        if you.startswith("/tokens"):
            try:
                max_new = int(you.split()[1])
                print(f"   (max new tokens = {max_new})")
            except (IndexError, ValueError):
                print("   usage: /tokens 200")
            continue
        if you.startswith("/prompt"):
            you = you[len("/prompt"):].strip()

        text = reply(
            model, tokenizer, you, device,
            max_new_tokens=max_new, temperature=temperature,
        )
        print(f"Guri > {text}\n")


def main() -> None:
    parser = argparse.ArgumentParser(description="Talk to your Guri brain")
    parser.add_argument("--ckpt", default="out/ckpt_latest.pt")
    parser.add_argument("--vocab", default=None)
    args = parser.parse_args()
    model, tokenizer, device = load_brain(args.ckpt, args.vocab)
    chat_loop(model, tokenizer, device)

In [ ]:
# One-shot example first (no typing needed):
import os
brain, vocab, dev = load_brain(
    os.path.join(OUT_DIR, "ckpt_latest.pt"),
    CFG["vocab_path"],
)
sample = reply(brain, vocab, "Once upon a time", dev, max_new_tokens=80)
print("Once upon a time" + sample)

In [ ]:
# Interactive chat — click in the box below, type, press Enter.
# Type /quit when you want to stop.
chat_loop(brain, vocab, dev)

## 🎓 What's next? (Level 2 preview)

Your Level 1 brain is a **baby storyteller**. To make it smarter:

1. **More food** — set `book_amount = "normal"` or add your own `.txt` files
2. **Longer training** — set `HOW_LONG = "long"` (resumes automatically)
3. **Bigger brain** — edit `n_layer` / `n_embd` in Step 4 (Level 2 = ~50M knobs)
4. **Save forever** — upload `OUT_DIR` to Hugging Face (see `GUIDE.md`)

Every level is trained **from scratch** — your brain, your rules. 🤝